# Logistic Regression & The Perceptron Trick

Logistic Regression is a foundational classification algorithm in machine learning. Before diving into the probabilistic loss-minimization framework of Logistic Regression, it is highly educational to understand its geometric ancestor: the **Perceptron**.

In these notes, we will explore:
1. **Introduction to Logistic Regression** (Core concepts, prerequisites, and limitations)
2. **The Perceptron Trick** (Geometric intuition, updating coefficients)
3. **Region Labeling** (Plugging in coordinates, classifying positive vs. negative regions)
4. **The Simplified Update Rule** (An elegant mathematical formulation for all update cases)
5. **Python Implementation from Scratch** (Generating data, coding the training loop)
6. **Visualization and Boundary Shifts** (Plotting decision boundaries)
7. **Perceptron vs. Logistic Regression** (Understanding why Perceptron isn't optimal and why Logistic Regression is superior)


## 1. Introduction to Logistic Regression

**Logistic Regression** is a fundamental supervised learning algorithm used for **binary classification** (predicting one of two classes, e.g., "Yes/No", "Green/Blue", "Spam/Not Spam"). 

### The Key Prerequisite: Linear Separability
For Logistic Regression (and the Perceptron) to work effectively, the dataset should be **linearly separable** (or mostly linearly separable). This means we can draw a straight boundary that partitions the two classes:
- In **2D space**, this boundary is a **straight line**.
- In **3D space**, this boundary is a **plane**.
- In **higher-dimensional space ($D > 3$)**, this boundary is a **hyperplane**.

### The Core Objective
The model's objective is to determine the coefficients/weights of this boundary to divide the feature space into two regions:
1. **Positive Class ($y = 1$)**
2. **Negative Class ($y = 0$)**

### Limitation: Non-Linear Data
If the classes cannot be separated by a straight line or plane (e.g., concentric circles, spiral patterns, or XOR configurations), simple Logistic Regression/Perceptron will fail to classify the data accurately. In such cases, non-linear classifiers (like Decision Trees, SVMs with non-linear kernels, or Deep Neural Networks) or feature transformations (Polynomial features) are required.

## 2. The Perceptron Trick

The **Perceptron** is a simple neural network unit. The "Perceptron Trick" is an intuitive, iterative geometric algorithm used to find a line that separates two classes.

### How It Works (Conceptual Steps):
1. **Initialize a Random Line:** Start with an arbitrary decision boundary defined by coefficients (weights and bias).
2. **Iterative Learning:**
   - Loop through the dataset or select a point at random.
   - Test if the model correctly classifies this point.
   - **If correct:** Do nothing.
   - **If incorrect (misclassified):** Apply a mathematical transformation to the coefficients (tilt or shift the line) to pull the boundary closer to the misclassified point.
3. **Repeat:** Continue this process until the line correctly classifies all points (convergence) or a maximum number of iterations (epochs) is reached.

### Geometric Shift: Transformation Logic
Suppose our line is represented in 2D space by:
$$Ax + By + C = 0$$

If we select a point $P(x_i, y_i)$ that is misclassified:
- To move the line **closer** to the point, we modify the coefficients $A, B$, and $C$.
- A direct update (adding/subtracting the coordinates) can cause the line to make massive, unstable swings, potentially misclassifying previously correct points.
- To prevent this, we introduce a **Learning Rate ($\eta$, e.g., $0.01$ or $0.1$)** to scale the updates down, ensuring the line adjusts gradually and stably.
- Specifically, for a misclassified positive point, we add the scaled coordinates: $A \leftarrow A + \eta x_i$, $B \leftarrow B + \eta y_i$, $C \leftarrow C + \eta$. For a misclassified negative point, we subtract them.

## 3. Labeling Regions and the Update Rule

### Region Labeling
To identify which side of the boundary $Ax + By + C = 0$ is the positive or negative region, we plug any coordinate $(x, y)$ into the line equation:
- If $Ax + By + C > 0$, the point lies in the **positive region** (typically labeled as $1$ or green).
- If $Ax + By + C < 0$, the point lies in the **negative region** (typically labeled as $0$ or blue).

### General Notation
We can generalize the line equation to higher dimensions:
$$w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n = 0$$

Using vector notation:
$$W^T X = 0$$

Where:
- $W = [w_0, w_1, w_2, \dots, w_n]^T$ is the weight vector (with $w_0$ being the bias $b$).
- $X = [1, x_1, x_2, \dots, x_n]^T$ is the feature vector (with $1$ prepended to handle the bias term).

The model predicts:
$$\hat{y} = \begin{cases} 
      1 & \text{if } W^T X \ge 0 \\
      0 & \text{if } W^T X < 0 
   \end{cases}$$

---

### The Unified Update Rule
Instead of checking multiple conditional statements for positive and negative misclassifications, we use a single, elegant formula to update the weights:

$$W_{\text{new}} = W_{\text{old}} + \eta \cdot (y_i - \hat{y}_i) \cdot X_i$$

Where:
- $y_i$ is the actual class label ($0$ or $1$).
- $\hat{y}_i$ is the predicted class label ($0$ or $1$).
- $\eta$ is the learning rate.
- $X_i$ is the input vector $[1, x_{i1}, x_{i2}]$.

#### Let's verify how this formula automatically handles all cases:

| Scenario | Actual $y_i$ | Predicted $\hat{y}_i$ | Classification | Term $(y_i - \hat{y}_i)$ | Resulting Update | Effect |
| :--- | :---: | :---: | :---: | :---: | :--- | :--- |
| **Case 1** | $1$ | $1$ | **Correct** | $1 - 1 = 0$ | $W_{\text{new}} = W_{\text{old}}$ | No change to weights. |
| **Case 2** | $0$ | $0$ | **Correct** | $0 - 0 = 0$ | $W_{\text{new}} = W_{\text{old}}$ | No change to weights. |
| **Case 3** | $1$ | $0$ | **Incorrect (False Negative)** | $1 - 0 = 1$ | $W_{\text{new}} = W_{\text{old}} + \eta X_i$ | Adds scaled features to weights; pulls the line toward the positive point. |
| **Case 4** | $0$ | $1$ | **Incorrect (False Positive)** | $0 - 1 = -1$ | $W_{\text{new}} = W_{\text{old}} - \eta X_i$ | Subtracts scaled features from weights; pushes the line away from the negative point. |

This formula is not only mathematically concise but also computationally efficient, as it avoids branching conditions in the training loop.

## 4. Setup and Libraries Import

First, let's set up our workspace and import the required scientific computing and visualization libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# Setting aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

## 5. Creating a Synthetic Linear Classification Dataset

We will generate a synthetic 2D binary classification dataset using Scikit-Learn's `make_classification`. Since the Perceptron requires linear separability, we will configure the parameters to ensure a clear separation.

In [ ]:
# Generate synthetic 2-class classification dataset
X, y = make_classification(
    n_samples=100, 
    n_features=2, 
    n_informative=2,
    n_redundant=0, 
    n_clusters_per_class=1, 
    class_sep=2.0, 
    random_state=42
)

# Plot the generated dataset
plt.figure(figsize=(10, 6))
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=80, label='Class 0 (Blue)')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=80, label='Class 1 (Green)')
plt.title('Synthetic 2D Binary Classification Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1 ($x_1$)', fontsize=12)
plt.ylabel('Feature 2 ($x_2$)', fontsize=12)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

## 6. Implementing the Perceptron Trick from Scratch

We will implement the `perceptron` function.
Inside this function:
1. We prepend a column of $1$s to $X$ to serve as the bias feature.
2. We initialize the weights $W$ with values of $1$.
3. We run the training loop for $1,000$ epochs.
4. In each epoch, we pick a random point, compute the dot product, make a step prediction ($1$ if result $\ge 0$, else $0$), and apply the unified weight update rule.
5. We also track the intermediate weights to visualize how the decision boundary evolves.

In [ ]:
def perceptron(X, y, epochs=1000, lr=0.01):
    # Prepend a column of 1s to X for the bias term
    X_bias = np.insert(X, 0, 1, axis=1)
    
    # Initialize weights: w0 (bias), w1, w2. Here we initialize with 1s.
    weights = np.ones(X_bias.shape[1])
    
    # List to store history of weights for animation/visualization
    weights_history = [weights.copy()]
    
    for epoch in range(epochs):
        # Select a random data point index
        i = np.random.randint(0, X_bias.shape[0])
        
        # Select the feature vector and target label for the chosen point
        x_i = X_bias[i]
        y_i = y[i]
        
        # Calculate dot product (W^T * X)
        dot_product = np.dot(weights, x_i)
        
        # Apply the step activation function
        y_hat = 1 if dot_product >= 0 else 0
        
        # Check if misclassified and update weights using the unified rule
        # Note: If correctly classified, y_i - y_hat = 0, so weights remain unchanged.
        if y_i != y_hat:
            weights = weights + lr * (y_i - y_hat) * x_i
            # Save the updated weights for visualization
            weights_history.append(weights.copy())
            
    return weights, np.array(weights_history)

# Train the perceptron
final_weights, weights_history = perceptron(X, y, epochs=1000, lr=0.01)

print("Training completed!")
print(f"Final Weights: {final_weights}")
print(f"Number of boundary shifts during training: {len(weights_history) - 1}")

## 7. Visualizing the Decision Boundary Evolution

The equation of our decision boundary is:
$$w_0 + w_1 x_1 + w_2 x_2 = 0$$

To plot this line in our 2D feature space ($x_1$ vs $x_2$), we solve for $x_2$:
$$w_2 x_2 = -w_1 x_1 - w_0 \implies x_2 = -\frac{w_1}{w_2} x_1 - \frac{w_0}{w_2}$$

Therefore:
- **Slope ($m$):** $-\frac{w_1}{w_2}$
- **Intercept ($c$):** $-\frac{w_0}{w_2}$

Let's write a function to plot the dataset along with the final decision boundary, and show the progression of boundaries as the algorithm learned.

In [ ]:
# Extract feature range for plotting
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x_plot = np.linspace(x_min, x_max, 100)

plt.figure(figsize=(12, 7))

# 1. Plot data points
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=80, label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=80, label='Class 1')

# 2. Plot intermediate decision boundaries to show evolution
# We will plot a subset of intermediate lines to avoid cluttering the plot
num_lines_to_plot = min(15, len(weights_history))
indices_to_plot = np.linspace(0, len(weights_history) - 1, num_lines_to_plot, dtype=int)

for idx in indices_to_plot[:-1]:
    w = weights_history[idx]
    # Check if w[2] is non-zero to avoid division by zero
    if w[2] != 0:
        slope = -w[1] / w[2]
        intercept = -w[0] / w[2]
        y_plot = slope * x_plot + intercept
        plt.plot(x_plot, y_plot, color='#e74c3c', alpha=0.3, linestyle='--')

# 3. Plot the final decision boundary
w_final = final_weights
if w_final[2] != 0:
    slope_final = -w_final[1] / w_final[2]
    intercept_final = -w_final[0] / w_final[2]
    y_plot_final = slope_final * x_plot + intercept_final
    plt.plot(x_plot, y_plot_final, color='#c0392b', linewidth=3, label='Final Perceptron Boundary')

plt.title('Perceptron Decision Boundary Evolution', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1 ($x_1$)', fontsize=12)
plt.ylabel('Feature 2 ($x_2$)', fontsize=12)
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.legend(frameon=True, facecolor='white', loc='best')
plt.tight_layout()
plt.show()

## 8. Comparison: Perceptron vs. Logistic Regression

To understand the difference, let's train Scikit-Learn's `LogisticRegression` on the same dataset and compare the final decision boundaries side-by-side.

In [ ]:
# 1. Fit Logistic Regression
lr_model = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)
lr_model.fit(X, y)

# Extract Logistic Regression parameters
# Equation: intercept + coef_[0]*x1 + coef_[1]*x2 = 0
lr_w0 = lr_model.intercept_[0]
lr_w1 = lr_model.coef_[0][0]
lr_w2 = lr_model.coef_[0][1]

print("Logistic Regression Parameters:")
print(f"Intercept (w0): {lr_w0:.4f}")
print(f"Coefficients (w1, w2): [{lr_w1:.4f}, {lr_w2:.4f}]")

In [ ]:
plt.figure(figsize=(14, 6))

# Subplot 1: Perceptron
plt.subplot(1, 2, 1)
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=70, label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=70, label='Class 1')
if final_weights[2] != 0:
    p_slope = -final_weights[1] / final_weights[2]
    p_intercept = -final_weights[0] / final_weights[2]
    plt.plot(x_plot, p_slope * x_plot + p_intercept, color='#e74c3c', linewidth=2.5, label='Perceptron')
plt.title('Custom Perceptron Decision Boundary', fontsize=12, fontweight='bold')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.legend(frameon=True, facecolor='white')

# Subplot 2: Logistic Regression
plt.subplot(1, 2, 2)
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=70, label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=70, label='Class 1')
if lr_w2 != 0:
    lr_slope = -lr_w1 / lr_w2
    lr_intercept = -lr_w0 / lr_w2
    plt.plot(x_plot, lr_slope * x_plot + lr_intercept, color='#9b59b6', linewidth=2.5, label='Logistic Regression')
plt.title('Scikit-Learn Logistic Regression Boundary', fontsize=12, fontweight='bold')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.legend(frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

## 9. Key Differences & Insights

### Why the Boundaries Differ
If you inspect the two plots above:
- The **Perceptron** boundary might be very close to one of the classes. This is because the Perceptron algorithm has a **hard stopping condition**: it stops updating as soon as all training points are correctly classified. It does not evaluate the *quality* or *distance* of the boundary from the data points.
- **Logistic Regression**, on the other hand, seeks to maximize the likelihood of the dataset. It uses the **Sigmoid function** to map inputs to probabilities and minimizes **Log Loss (Cross-Entropy Loss)**. This forces the decision boundary to be positioned symmetrically, maximizing the margins between the two classes.

### Summary Comparison: Perceptron vs. Logistic Regression

| Feature | Perceptron Trick | Logistic Regression |
| :--- | :--- | :--- |
| **Output Type** | Hard Binary Prediction ($0$ or $1$ via step function) | Probabilistic Output (Probability between $0$ and $1$ via Sigmoid) |
| **Optimization Goal** | Find *any* separating line (satisfiability) | Find the *optimal* separating line with maximum margin (likelihood maximization) |
| **Stopping Condition** | Stops as soon as all training points are classified correctly | Continues optimization until loss is minimized and converges |
| **Robustness** | Sensitive to initial weights and selection order; prone to overfitting | Robust, generalizes well because it maximizes margins |
| **Handling Overlap** | Can oscillate endlessly if data is not linearly separable (requires a hard epoch limit or patience) | Handles overlapping/noisy classes naturally by outputting confidence/probabilities |

## Summary Checklist for Perceptron and Logistic Regression

1. **Verify Linear Separability:** Ensure the target classes can be separated by a linear hyperplane.
2. **Apply the Unified Update Rule:** Use $W \leftarrow W + \eta(y - \hat{y})X$ to update weights concisely.
3. **Control Steps with Learning Rate:** Set a small $\eta$ (e.g., $0.01$) to prevent unstable changes to the line position.
4. **Use Logistic Regression for Production:** Always favor Logistic Regression over the raw Perceptron because it optimizes the margin and outputs probability distributions rather than hard boundaries.